# MwC 02 — subject-level 26-ROI extraction

Extract and export the 12-cortical + 14-limbic subject-level MwC values.

- Formal analysis universe: 12 cortical + 14 limbic ROIs (26 nodes)
- Cross-network connectivity: 12 × 14 = 168 edges
- Pipeline: `mwc`; step 2 of 5
- Run mode: standalone; all prerequisite steps load automatically in the same kernel.


In [1]:
from pathlib import Path
import os

_repo = Path(os.environ.get("METAFL_ROOT", Path.cwd())).resolve()
if not (_repo / "pipeline_steps").is_dir() and (_repo.parent / "pipeline_steps").is_dir():
    _repo = _repo.parent
if not (_repo / "pipeline_steps").is_dir():
    raise FileNotFoundError("Set METAFL_ROOT to the repository root")
os.environ["METAFL_ROOT"] = str(_repo)
os.chdir(_repo)

def _run_step(relative_path):
    _path = _repo / "pipeline_steps" / relative_path
    _code = compile(_path.read_text(encoding="utf-8"), str(_path), "exec")
    exec(_code, get_ipython().user_ns)
_run_step("mwc_01_setup_recompute.py")


,input,size_kib
0,03_compute_normative_sc_mwc.py,12.4
1,session_gcmi_pet_AAL3_144.npz,17095.9
2,normative_SC_top10_mask.npz,53.7
3,subject_level_corticolimbic26_normativeSC10_Mw...,120.9
4,subject_level_AAL3_144_normativeSC10_MwC_wide....,288.0
5,validation_report.json,0.2
6,corticolimbic_12x14_ROI_definition.csv,0.6
7,clinical_data.xlsx,10.9
8,subject_level_corticolimbic26_age_sex_adjusted...,133.5


{
  "full_144": "/Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007/outputs/mwc/recomputed_normative_sc/subject_level_AAL3_144_normativeSC10_MwC_wide.csv.gz",
  "selected_26": "/Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007/outputs/mwc/recomputed_normative_sc/subject_level_corticolimbic26_normativeSC10_MwC_wide.csv",
  "validation": "/Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007/outputs/mwc/recomputed_normative_sc/calculation_validation_report.json"
}
PASS: AAL3-144 normative-SC-constrained MwC recomputed before 26-ROI extraction


,check,passed
0,sessions_253,True
1,retained_AAL3_nodes_144,True
2,fixed_dTOR_normative_SC_edges_1030,True
3,gcmi_floor_bits_0p01,True
4,row_normalization_error_below_1e_12,True
5,all_144_nodes_computed_before_26_extraction,True
6,isolated_nodes_remain_missing,True
7,full_144_matches_validated_reference,True
8,selected_26_matches_validated_reference,True


## Preamble


In [2]:
"""Auto-extracted pipeline step: MwC 02 — subject-level 26-ROI extraction.

Do not edit generated copies independently of the source notebooks.
"""


'Auto-extracted pipeline step: MwC 02 — subject-level 26-ROI extraction.\n\nDo not edit generated copies independently of the source notebooks.\n'

## Source 6


In [3]:
roi = pd.read_csv(ROI_FILE).reset_index(drop=True)
roi["node_index"] = np.arange(1, len(roi) + 1)
roi["ROI_full"] = [f"AAL3_{int(row.AAL3_ID):03d}_{row.ROI}" for row in roi.itertuples()]
assert len(roi) == N_ROIS

with AAL_VALIDATION_FILE.open(encoding="utf-8") as stream:
    aal_validation = json.load(stream)
assert aal_validation["status"] == "PASS"
assert aal_validation["checks"]["retained_nodes_144"]
assert aal_validation["checks"]["fixed_structural_edges_1030"]
assert mwc_calculation_validation["checks"]["all_144_nodes_computed_before_26_extraction"]

mwc_source = pd.read_csv(MWC_WIDE_FILE)
full_source = pd.read_csv(MWC_FULL_FILE)
assert len(mwc_source) == len(full_source) == 253
key_columns = ["subject_key", "session"]
assert mwc_source[key_columns].equals(full_source[key_columns])

mwc_columns = [f"MwC_{name}" for name in roi.ROI]
full_columns = [
    f"MwC_AAL3_{int(aal3_id):03d}_{name}"
    for aal3_id, name in zip(roi.AAL3_ID, roi.ROI)
]
assert set(mwc_columns).issubset(mwc_source)
assert set(full_columns).issubset(full_source)
all144_extraction_error = float(np.nanmax(np.abs(
    mwc_source[mwc_columns].to_numpy(float)
    - full_source[full_columns].to_numpy(float)
)))
assert all144_extraction_error < 1e-10


def _metadata_norm_key(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    match = re.fullmatch(r'P(\d{3})', text)
    if not match:
        raise ValueError(f'Expected a deidentified P### subject key, got {text!r}')
    return f'P{int(match.group(1)):03d}'


clinical_all_mwc = pd.read_excel(CLINICAL_FILE)
clinical_all_mwc['subject_key'] = clinical_all_mwc['subject_key'].map(_metadata_norm_key)
clinical_all_mwc['age'] = pd.to_numeric(clinical_all_mwc['age'], errors='coerce')
clinical_all_mwc['sex'] = pd.to_numeric(clinical_all_mwc['sex'], errors='coerce')
clinical_all_mwc['education'] = pd.to_numeric(clinical_all_mwc['education_years'], errors='coerce')
clinical_all_mwc['hdrs_pre'] = pd.to_numeric(clinical_all_mwc['hdrs17_pre'], errors='coerce')
clinical_all_mwc['hdrs_post'] = pd.to_numeric(clinical_all_mwc['hdrs17_post'], errors='coerce')
clinical_lookup = clinical_all_mwc.set_index('subject_key')
expected_clinical_counts = {"HC": 23, "nTRD": 25, "TRD1": 35, "TRD2": 85}
assert clinical_all_mwc["analysis_cohort"].value_counts().to_dict() == expected_clinical_counts

mwc_source = mwc_source.copy()
mwc_source['subject_key'] = mwc_source['subject_key'].map(_metadata_norm_key)
mwc_source.insert(0, "session_row", np.arange(len(mwc_source), dtype=int))
clinical_session_mask = mwc_source["subject_key"].isin(clinical_lookup.index)
mwc_source = mwc_source.loc[clinical_session_mask].reset_index(drop=True)

for column in ['analysis_cohort', 'analysis_arm', 'age', 'sex', 'education', 'hdrs_pre', 'hdrs_post']:
    mapped = mwc_source['subject_key'].map(clinical_lookup[column])
    mwc_source[column] = mapped

metadata_columns = [
    "session_row", "subject_key", "session", "analysis_cohort", "analysis_arm",
    "age", "sex", "education", "hdrs_pre", "hdrs_post",
]
mwc_wide = mwc_source[metadata_columns + mwc_columns].copy()
observed_node_mwc = mwc_wide[mwc_columns].to_numpy(float)

matrix_frame = pd.DataFrame(
    observed_node_mwc, index=mwc_wide.session_row, columns=roi.node_index,
)
matrix_frame.index.name = "session_row"
mwc_long = (
    matrix_frame.reset_index()
    .melt(id_vars="session_row", var_name="node_index", value_name="MwC")
    .merge(
        roi[["node_index", "network", "AAL3_ID", "ROI", "ROI_full"]],
        on="node_index", how="left", validate="many_to_one",
    )
    .merge(mwc_wide[metadata_columns], on="session_row", how="left", validate="many_to_one")
    .sort_values(["session_row", "node_index"])
    .reset_index(drop=True)
)

ALIASES = {
    "Frontal_Sup_Medial": "dmPFC",
    "Frontal_Med_Orb": "vmPFC",
    "OFCmed": "medOFC",
}


def display_roi_name(roi_name: str) -> str:
    for source_name, alias in ALIASES.items():
        if roi_name == source_name or roi_name.startswith(source_name + "_"):
            return roi_name.replace(source_name, alias, 1)
    return roi_name


mwc_long["display_roi"] = mwc_long.ROI.map(display_roi_name)
mwc_wide.to_csv(OUTPUT_DIR / "subject_level_corticolimbic26_normativeSC10_MwC_wide.csv", index=False)
mwc_long.to_csv(OUTPUT_DIR / "subject_level_corticolimbic26_normativeSC10_MwC_long.csv", index=False)

# HC-referenced deviations are regenerated from the normative-SC-constrained MwC values.
baseline_hc = mwc_wide.loc[
    mwc_wide.session.eq("ses-01") & mwc_wide.analysis_cohort.eq("HC")
].copy()
assert len(baseline_hc) == 23
hc_mean = baseline_hc[mwc_columns].mean()
hc_sd = baseline_hc[mwc_columns].std(ddof=1)
assert np.isfinite(hc_sd).all() and (hc_sd > 0).all()
baseline_groups = mwc_wide.loc[
    mwc_wide.session.eq("ses-01")
    & mwc_wide.analysis_cohort.isin(["nTRD", "TRD1"])
].copy()
assert baseline_groups.groupby("analysis_cohort").size().to_dict() == {"TRD1": 35, "nTRD": 25}
deviation = baseline_groups[[
    "session_row", "subject_key", "session", "analysis_cohort", "analysis_arm",
    "age", "sex", "education",
]].copy()
for source_column, roi_name in zip(mwc_columns, roi.ROI):
    deviation[f"MwC_deviation_{roi_name}"] = (
        baseline_groups[source_column].to_numpy(float) - hc_mean[source_column]
    ) / hc_sd[source_column]
deviation["display_group"] = deviation.analysis_cohort.map({"nTRD": "nTRD", "TRD1": "TRD"})
deviation.to_csv(
    OUTPUT_DIR / "subject_level_nTRD_TRD1_corticolimbic26_normativeSC10_MwC_HC_reference_deviation.csv",
    index=False,
)

display(pd.DataFrame({
    "quantity": [
        "upstream paired sessions", "clinical sessions", "ROIs per session",
        "long-table rows", "all-144 to 26 extraction maximum absolute error",
    ],
    "value": [253, len(mwc_wide), N_ROIS, len(mwc_long), all144_extraction_error],
}))
display(roi[["node_index", "network", "AAL3_ID", "ROI"]])


,quantity,value
0,upstream paired sessions,253.0
1,clinical sessions,253.0
2,ROIs per session,26.0
3,long-table rows,6578.0
4,all-144 to 26 extraction maximum absolute error,0.0


,node_index,network,AAL3_ID,ROI
0,1,cortical,3,Frontal_Sup_2_L
1,2,cortical,4,Frontal_Sup_2_R
2,3,cortical,5,Frontal_Mid_2_L
3,4,cortical,6,Frontal_Mid_2_R
4,5,cortical,19,Frontal_Sup_Medial_L
5,6,cortical,20,Frontal_Sup_Medial_R
6,7,cortical,21,Frontal_Med_Orb_L
7,8,cortical,22,Frontal_Med_Orb_R
8,9,cortical,25,OFCmed_L
9,10,cortical,26,OFCmed_R


## Results preview

Key numerical tables and figures generated by this step are displayed below. The full-resolution files remain under `outputs/`.


In [4]:
from io import BytesIO
import json as _preview_json
import pandas as _preview_pd
from PIL import Image as PILImage
from IPython.display import Image as NotebookImage, Markdown, display

_inline_table_specs = [{'title': 'Subject-level 26-ROI MwC values', 'path': 'outputs/mwc/subject_level_corticolimbic26_normativeSC10_MwC_wide.csv', 'max_rows': 10}]
_inline_figure_paths = []

display(Markdown("### Key tables"))
for _table_spec in _inline_table_specs:
    _relative_table = _table_spec["path"]
    _table_path = _repo / _relative_table
    if not _table_path.is_file():
        raise FileNotFoundError(_table_path)
    if _table_path.suffix.lower() == ".csv":
        _preview_table = _preview_pd.read_csv(_table_path)
    elif _table_path.suffix.lower() in {".xlsx", ".xls"}:
        _preview_table = _preview_pd.read_excel(
            _table_path, sheet_name=_table_spec.get("sheet", 0)
        )
    elif _table_path.suffix.lower() == ".json":
        _payload = _preview_json.loads(_table_path.read_text(encoding="utf-8"))
        _checks = _payload.get("checks", _payload)
        _preview_table = _preview_pd.DataFrame(
            {"item": list(_checks), "value": list(_checks.values())}
        ) if isinstance(_checks, dict) else _preview_pd.json_normalize(_payload)
    else:
        raise ValueError(f"Unsupported table format: {_table_path.suffix}")
    if _table_spec.get("query"):
        _preview_table = _preview_table.query(_table_spec["query"]).reset_index(drop=True)
    _max_rows = int(_table_spec.get("max_rows", 40))
    display(Markdown(
        f"**{_table_spec['title']}**  \n"
        f"`{_relative_table}` — {len(_preview_table)} rows × {len(_preview_table.columns)} columns"
    ))
    with _preview_pd.option_context("display.max_columns", 40, "display.max_colwidth", 60):
        display(_preview_table.head(_max_rows))

if _inline_figure_paths:
    display(Markdown("### Figures"))
for _relative_figure in _inline_figure_paths:
    _figure_path = _repo / _relative_figure
    if not _figure_path.is_file():
        raise FileNotFoundError(_figure_path)
    with PILImage.open(_figure_path) as _source_image:
        _preview_image = _source_image.convert("RGB")
        _preview_image.thumbnail((1600, 1200), PILImage.Resampling.LANCZOS)
        _preview_buffer = BytesIO()
        _preview_image.save(_preview_buffer, format="PNG", optimize=True)
    display(Markdown(f"**`{_relative_figure}`**"))
    display(NotebookImage(data=_preview_buffer.getvalue(), format="png"))


### Key tables

**Subject-level 26-ROI MwC values**  
`outputs/mwc/subject_level_corticolimbic26_normativeSC10_MwC_wide.csv` — 253 rows × 36 columns

,session_row,subject_key,session,analysis_cohort,analysis_arm,age,sex,education,hdrs_pre,hdrs_post,MwC_Frontal_Sup_2_L,MwC_Frontal_Sup_2_R,MwC_Frontal_Mid_2_L,MwC_Frontal_Mid_2_R,MwC_Frontal_Sup_Medial_L,MwC_Frontal_Sup_Medial_R,MwC_Frontal_Med_Orb_L,MwC_Frontal_Med_Orb_R,MwC_OFCmed_L,MwC_OFCmed_R,MwC_OFClat_L,MwC_OFClat_R,MwC_Hippocampus_L,MwC_Hippocampus_R,MwC_ParaHippocampal_L,MwC_ParaHippocampal_R,MwC_Amygdala_L,MwC_Amygdala_R,MwC_Caudate_L,MwC_Caudate_R,MwC_Putamen_L,MwC_Putamen_R,MwC_Pallidum_L,MwC_Pallidum_R,MwC_ACC_sub_L,MwC_ACC_sub_R
0,0,P001,ses-01,HC,NaN,59,1,12,2,NaN,104.513317,105.916085,101.525201,103.553667,109.682251,110.192499,101.677325,99.995172,91.392529,84.180175,104.062184,99.929050,81.259752,84.292821,78.846692,80.141912,72.476073,80.556167,101.537162,104.150367,104.128549,103.393670,106.469384,110.510283,103.183552,107.687699
1,1,P002,ses-01,HC,NaN,62,1,12,6,NaN,102.187466,103.006200,103.599937,103.022101,103.169599,104.012152,99.138568,99.126566,90.035491,91.231038,98.504059,95.570785,90.720997,88.804402,93.171786,91.420706,74.383393,78.708852,101.781964,97.817338,98.483566,102.387214,99.006958,104.132817,96.241015,101.358509
2,2,P003,ses-01,HC,NaN,69,1,13,5,NaN,99.435309,100.379242,101.393215,103.724988,101.276105,100.856174,93.183206,93.842440,89.140821,87.982422,100.307417,101.857924,83.458607,81.221666,85.598282,81.547573,77.943026,79.300878,104.849017,100.011357,96.764072,105.846208,118.923253,113.841691,93.906571,96.420437
3,3,P004,ses-01,HC,NaN,68,1,6,0,NaN,107.491583,103.816042,105.734680,101.758798,109.509176,106.699322,101.731081,99.817077,89.150441,84.056528,92.109750,93.838137,84.559235,87.519203,77.241440,82.781601,77.793485,101.402256,110.183555,100.170017,104.442371,104.033039,115.828172,109.560579,105.169144,NaN
4,4,P005,ses-01,HC,NaN,78,1,6,0,NaN,99.520538,99.561391,103.457519,102.057378,99.206702,99.854082,95.399528,96.134915,95.578380,102.706058,96.437866,104.180801,73.438197,81.224668,79.194168,93.027027,75.450695,77.645421,104.290736,98.601287,100.155233,104.490777,108.846307,115.706508,102.782343,101.880801
5,5,P006,ses-01,HC,NaN,58,0,12,7,NaN,102.631716,103.793775,105.103743,106.696430,101.335131,100.830058,98.317441,93.325514,102.627533,98.720858,100.933812,102.608800,81.062130,81.109807,85.160948,88.259857,79.461788,85.486199,103.904686,102.817952,100.731951,107.999500,99.482762,107.260946,107.536978,100.282426
6,6,P007,ses-01,HC,NaN,53,1,14,5,NaN,101.775816,101.964323,105.990383,102.886331,104.775567,102.536132,103.692521,105.958685,109.071495,100.434506,100.846266,102.950776,96.624541,93.415825,96.232994,86.524513,74.605709,92.082715,97.389776,101.770730,97.660976,106.142752,109.409887,111.328696,106.147721,89.877207
7,7,P008,ses-01,HC,NaN,46,1,14,0,NaN,102.259155,104.321825,107.960896,105.896368,103.433693,105.287399,101.249254,100.327826,106.467022,101.004672,103.775209,102.210081,93.248839,88.438740,87.401623,88.318543,84.856203,88.818510,107.296974,100.589998,105.660816,108.976498,123.827042,111.723393,109.537813,109.084191
8,8,P009,ses-01,HC,NaN,59,1,12,0,NaN,101.424250,102.232122,105.531314,104.588198,111.405251,107.442335,100.121271,99.173152,96.461813,95.375836,100.270576,105.117500,86.480469,80.015963,78.764697,82.159830,90.369912,86.554814,102.750209,102.431293,104.424501,105.506933,116.002536,113.239982,101.790785,101.855556
9,9,P010,ses-01,HC,NaN,38,1,16,1,NaN,105.980913,106.057209,106.922943,104.746620,105.250751,106.224540,96.798521,99.794802,105.539443,101.054352,107.932199,NaN,93.065254,89.037088,92.030108,90.798895,86.639222,86.155953,108.033751,103.267060,104.615943,102.124473,94.652012,109.693709,112.040492,108.928335
